# 22_diagnostic_const_baseline.ipynb

Supplementary к `21_diagnostic_error_maps.ipynb`. Отвечает на два конкретных
вопроса, которые остались открытыми после первой диагностики.

## Вопрос 1 — это правда «модель сваливается в константу»?

На error maps GeoFNO (кластер B/hard) было видно равномерно жёлтое пятно по всей
внутренней области фантома. Это похоже на «модель предсказывает усреднённую
карту». Здесь сравниваем модель с тремя константными baseline'ами **только
внутри маски фантома** (фон везде угадан правильно):

- **oracle_mean**: предсказание = среднее target внутри маски (best constant
  если ты заранее знаешь target).
- **bg_1500**: предсказание = 1500 м/с везде. Если модель близка к этому —
  она вообще не «вошла» в фантом.
- **train_mean**: предсказание = глобальное среднее train internal-mask.
  Если модель близка к этому — она угадывает датасетное среднее, игнорируя вход.

Если модель работает хуже oracle_mean — она «знает меньше, чем константа равная
истинному среднему», то есть **не извлекает структуру** и заменяет её на
смещённое константное предсказание.

## Вопрос 2 — а спектр всё-таки architecture- или physics-bound?

Старая декомпозиция считалась на полной error map, и фон (90% площади)
доминировал в Фурье. Здесь пересчитываем спектр **только внутри маски**, причём
после обнуления фона. Это даёт реальное распределение ошибки по частотам без
artifact'а от однородного фона.

## Требования к kernel

Этот ноутбук переиспользует объекты из `21_diagnostic_error_maps.ipynb`. Запускай
его **в том же kernel'е**, после того как 21-й полностью прошёл. Если хочешь
независимый запуск — скопируй подготовительные ячейки 1-7 из 21-го сюда.

## 1. Sanity check

In [2]:
# These come from 21_diagnostic_error_maps.ipynb. If any are missing — что-то
# пошло не так с kernel state.
required_objects = ["models", "load_sample", "stats", "test_files",
                    "sample_clusters", "cluster_names", "M_hom_complex",
                    "agg", "H", "W", "out_dir"]
missing = [o for o in required_objects if o not in dir()]
assert not missing, f"Missing kernel objects: {missing}. Re-run 21_diagnostic_error_maps.ipynb first."
print("All kernel objects available.")
print("models loaded:", list(models.keys()))

AssertionError: Missing kernel objects: ['models', 'load_sample', 'stats', 'test_files', 'sample_clusters', 'cluster_names', 'M_hom_complex', 'agg', 'H', 'W', 'out_dir']. Re-run 21_diagnostic_error_maps.ipynb first.

## 2. Phantom mask и `train_mean` константа

Маска: пиксели, где target существенно отличается от 1500 м/с (фон). Используем
threshold=10 м/с — он мягкий и не порежет тонкие структуры по краям, но
исключит чистый фон.

`train_mean`: среднее target внутри маски, усреднённое по train. Это та
«константа без знаний о конкретном входе», которую сеть **могла бы выучить как
оптимальную точку** при провале backward signal.

In [ ]:
def phantom_mask(c_phys, threshold=10.0, background_value=1500.0):
    """Returns boolean (H, W) mask of pixels inside the phantom (not background)."""
    return np.abs(c_phys - background_value) > threshold

# Compute train_mean once
print("Computing train_mean of in-mask values...")
train_mask_means = []
train_files_local = sorted((Path(DIAG["dataset_root"]) / "train").glob("*.npz"))
for p in train_files_local:
    c = np.load(p)["c"].astype(np.float32)
    mk = phantom_mask(c)
    if mk.sum() > 0:
        train_mask_means.append(float(c[mk].mean()))
TRAIN_MEAN = float(np.mean(train_mask_means))
print(f"train_mean (mean of in-phantom values across train): {TRAIN_MEAN:.2f} m/s")

# Quick visual check on one sample
sample0 = load_sample(test_files[0], stats["c_min"], stats["c_max"], M_hom_complex)
mk0 = phantom_mask(sample0["c_phys"])
print(f"\nSample 0: phantom area = {int(mk0.sum())} pixels = {mk0.sum() / (H*W) * 100:.1f}% of grid")
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(sample0["c_phys"], origin="lower"); ax[0].set_title("target")
ax[1].imshow(mk0, origin="lower", cmap="gray"); ax[1].set_title(f"phantom mask")
plt.tight_layout(); plt.show()

## 3. Constant-prediction baselines vs модель

Для каждой модели и каждого кластера считаем **на трудных примерах** (top-25%
по RMSE) средние значения четырёх метрик:

- `model_rmse_in_mask`: RMSE модели внутри маски
- `oracle_const_rmse`: RMSE при предсказании = mean(target в маске) (best const)
- `bg_const_rmse`: RMSE при предсказании = 1500 везде
- `train_const_rmse`: RMSE при предсказании = train_mean везде в маске

Также считаем **bias**: насколько среднее предсказания внутри маски смещено от
среднего target внутри маски. Если bias близок к нулю — модель «правильно
угадала уровень», но не разрешила структуру. Если bias большой и отрицательный
— она «забралась к фону».

In [ ]:
def in_mask_rmse(pred, target, mask):
    if mask.sum() == 0:
        return 0.0
    diff = (pred - target)[mask]
    return float(np.sqrt(np.mean(diff ** 2)))

def constant_baselines_for_sample(target, pred, mask, train_mean, background_value=1500.0):
    """Returns dict of in-mask RMSEs for the model and three const baselines."""
    if mask.sum() == 0:
        return None
    target_mean = float(target[mask].mean())
    pred_mean = float(pred[mask].mean())

    return {
        "model_rmse":          in_mask_rmse(pred, target, mask),
        "oracle_const_rmse":   in_mask_rmse(np.full_like(target, target_mean), target, mask),
        "bg_const_rmse":       in_mask_rmse(np.full_like(target, background_value), target, mask),
        "train_const_rmse":    in_mask_rmse(np.full_like(target, train_mean), target, mask),
        "target_mean":         target_mean,
        "pred_mean":           pred_mean,
        "bias":                pred_mean - target_mean,
    }

# Iterate over hard examples (top-25%) per cluster, per model
const_results = {name: {} for name in models.keys()}
for name in models.keys():
    mdl, predict_fn = models[name]
    metrics_list = agg[name]["per_sample"]
    rmses = np.array([m["rmse"] for m in metrics_list])
    for cn in cluster_names:
        idxs = np.where(sample_clusters == cn)[0]
        if len(idxs) == 0:
            continue
        rmses_in = rmses[idxs]
        hard_thresh = np.quantile(rmses_in, DIAG["hard_quantile"])
        hard_idx = idxs[rmses_in >= hard_thresh]

        rows = []
        for i in hard_idx:
            sample = load_sample(test_files[i], stats["c_min"], stats["c_max"], M_hom_complex)
            pred = predict_fn(mdl, sample)
            target = sample["c_phys"]
            mk = phantom_mask(target)
            r = constant_baselines_for_sample(target, pred, mk, TRAIN_MEAN)
            if r is not None:
                rows.append(r)

        if not rows:
            continue
        const_results[name][cn] = {
            k: float(np.mean([r[k] for r in rows]))
            for k in rows[0].keys()
        }
        const_results[name][cn]["n"] = len(rows)

# Pretty-print
print("\n=== In-mask RMSE on HARD examples (top-25% by RMSE) ===")
print("  Compare model to constant baselines.  All values in m/s.\n")
print("  oracle = mean(target inside mask)  |  bg = 1500 everywhere  |  train = global train_mean")
print()
header = f"  {'model':<12}{'cluster':<6}{'n':>4}{'model':>10}{'oracle':>10}{'bg=1500':>10}{'train':>10}{'bias':>10}"
print(header)
print("  " + "-" * (len(header) - 2))
for name in models.keys():
    for cn in cluster_names:
        r = const_results[name].get(cn)
        if r is None:
            continue
        print(f"  {name:<12}{cn:<6}{r['n']:>4}{r['model_rmse']:>10.2f}"
              f"{r['oracle_const_rmse']:>10.2f}{r['bg_const_rmse']:>10.2f}"
              f"{r['train_const_rmse']:>10.2f}{r['bias']:>+10.2f}")
    print()

with open(out_dir / "const_baseline_comparison.json", "w", encoding="utf-8") as f:
    json.dump(const_results, f, ensure_ascii=False, indent=2)
print(f"saved to {out_dir / 'const_baseline_comparison.json'}")

### 3.1 Auto-interpretation

Ниже автоматическая интерпретация. Считаем «модель сваливается в константу», если
`model_rmse / oracle_const_rmse > 0.95` — то есть модель работает не лучше, чем
если бы просто угадывала среднее.

In [ ]:
print("\n=== Auto-interpretation ===\n")
print("  Verdict per (model, cluster) on HARD subset:")
print("  - model_rmse / oracle_const < 0.7  → model adds real structure beyond constant")
print("  - 0.7 .. 0.95                     → partial structure, weak model")
print("  - > 0.95                           → effectively constant prediction (collapse)")
print()
for name in models.keys():
    print(f"  {name}")
    for cn in cluster_names:
        r = const_results[name].get(cn)
        if r is None:
            continue
        ratio = r["model_rmse"] / max(r["oracle_const_rmse"], 1e-6)
        bias = r["bias"]
        if ratio > 0.95:
            verdict = "COLLAPSE: model ≈ constant"
        elif ratio > 0.7:
            verdict = "weak: partial structure recovery"
        else:
            verdict = "OK: real structure"
        bias_note = f"  (bias={bias:+.1f} m/s)"
        print(f"    cluster {cn}:  model/oracle = {ratio:.2f}   {verdict}{bias_note}")
    print()

## 4. Bar plot: модель vs const-baseline'ы

Группированный bar plot по кластерам. Высота столбца — in-mask RMSE на hard
subset.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 4.5), sharey=True)
if len(models) == 1:
    axes = [axes]

x_labels = cluster_names
n_clusters = len(x_labels)
xs = np.arange(n_clusters)
width = 0.20

for ax, name in zip(axes, models.keys()):
    vals_model = []
    vals_oracle = []
    vals_bg = []
    vals_train = []
    for cn in cluster_names:
        r = const_results[name].get(cn)
        if r is None:
            for L in (vals_model, vals_oracle, vals_bg, vals_train):
                L.append(0)
        else:
            vals_model.append(r["model_rmse"])
            vals_oracle.append(r["oracle_const_rmse"])
            vals_bg.append(r["bg_const_rmse"])
            vals_train.append(r["train_const_rmse"])

    ax.bar(xs - 1.5 * width, vals_model, width, label="model")
    ax.bar(xs - 0.5 * width, vals_oracle, width, label="oracle const")
    ax.bar(xs + 0.5 * width, vals_bg, width, label="bg=1500")
    ax.bar(xs + 1.5 * width, vals_train, width, label="train mean")

    ax.set_xticks(xs)
    ax.set_xticklabels(x_labels)
    ax.set_title(name)
    ax.set_ylabel("in-mask RMSE (m/s)")
    ax.legend(fontsize=8, loc="upper left")
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Hard examples: model vs constant baselines (in-mask RMSE)", fontsize=12)
plt.tight_layout()
fig_path = out_dir / "const_baseline_comparison.png"
plt.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"saved to {fig_path}")

## 5. Spectral decomposition **внутри маски фантома**

Та же декомпозиция, что в 21-м, но вход в FFT — error map с **обнулённым
фоном**. Это убирает доминирование однородного фона в Фурье и показывает
реальную частотную структуру ошибки внутри объекта.

In [ ]:
def freq_band_energy_masked(err_map, mask, low_band, mid_band):
    """Spectral energy fractions of (err_map * mask) — background zeroed out."""
    masked = err_map * mask
    F = np.fft.rfft2(masked)
    Hf, Wf = F.shape
    ky = np.fft.fftfreq(Hf, d=1.0) * Hf
    kx = np.arange(Wf)
    KY, KX = np.meshgrid(ky, kx, indexing="ij")
    K = np.sqrt(KY * KY + KX * KX)

    energy = np.abs(F) ** 2
    total = energy.sum()
    if total < 1e-12:
        return {"low": 0.0, "mid": 0.0, "high": 0.0, "total": 0.0}
    e_low  = energy[K <= low_band].sum()
    e_mid  = energy[(K > low_band) & (K <= mid_band)].sum()
    e_high = energy[K > mid_band].sum()
    return {"low": float(e_low / total), "mid": float(e_mid / total),
            "high": float(e_high / total), "total": float(total)}


# Recompute spectral decomposition on cluster-averaged hard error maps,
# but masked. Re-use cluster_error_maps from notebook 21.
masked_spec = {}
print("\n=== Masked spectral decomposition (background pixels zeroed) ===\n")

# We need an average phantom mask per cluster. Compute it as union of masks
# for the hard examples used to build cluster_error_maps[name][cluster]["hard_map"].
# For simplicity, take a single representative mask = mask of average-target.
for name in models.keys():
    if name not in cluster_error_maps:
        continue
    masked_spec[name] = {}
    print(f"  {name}")
    print(f"    {'cluster':<10}{'easy_low':>9}{'easy_mid':>9}{'easy_high':>10}"
          f"   {'hard_low':>9}{'hard_mid':>9}{'hard_high':>10}")
    for cn, maps in cluster_error_maps[name].items():
        # masks built from cluster-averaged target maps
        mask_easy = phantom_mask(maps["target_easy_avg"])
        mask_hard = phantom_mask(maps["target_hard_avg"])
        be = freq_band_energy_masked(maps["easy_map"], mask_easy,
                                     DIAG["freq_band_low"], DIAG["freq_band_mid"])
        bh = freq_band_energy_masked(maps["hard_map"], mask_hard,
                                     DIAG["freq_band_low"], DIAG["freq_band_mid"])
        masked_spec[name][cn] = {"easy": be, "hard": bh}
        print(f"    {cn:<10}{be['low']:9.3f}{be['mid']:9.3f}{be['high']:10.3f}"
              f"   {bh['low']:9.3f}{bh['mid']:9.3f}{bh['high']:10.3f}")
    print()

with open(out_dir / "spectral_decomposition_masked.json", "w", encoding="utf-8") as f:
    json.dump(masked_spec, f, ensure_ascii=False, indent=2)
print(f"saved to {out_dir / 'spectral_decomposition_masked.json'}")

In [ ]:
# Updated heuristic interpretation on the masked spectrum
print("\n=== Diagnostic interpretation (MASKED spectral) ===\n")
print("  Fraction of HIGH-frequency error energy (|k| > {}):".format(DIAG["freq_band_mid"]))
print("  - high fraction (>0.4)  → modes/capacity ceiling (architecture-bound)")
print("  - low fraction  (<0.15) → forward-operator nonlinearity / mode collapse (physics-bound)")
print()
for name in models.keys():
    if name not in masked_spec:
        continue
    high_easy_avg = np.mean([masked_spec[name][cn]["easy"]["high"] for cn in masked_spec[name]])
    high_hard_avg = np.mean([masked_spec[name][cn]["hard"]["high"] for cn in masked_spec[name]])
    if high_hard_avg > 0.4:
        verdict = "HIGH-band → architecture-bound (more modes / less aggressive bottleneck)"
    elif high_hard_avg < 0.15:
        verdict = "LOW/MID-band → physics-bound (multi-frequency, unrolling) or model collapse"
    else:
        verdict = "mixed regime"
    print(f"  {name:<14}  easy_high={high_easy_avg:.2f}  hard_high={high_hard_avg:.2f}   →  {verdict}")